# **CCB03. Determining relative protein abundance in *Cyclospora cayetanensis* using orthologs from *Toxoplasma gondii*.**

**Date**: Oct 10, 2026

**TL;DR**
- **Goal**: Determine the relative steady-state protein abundances (SSAs) in C. cayetanensis (CCay) to find a strong candidate for a rapid antigen test target.
- **Problem**: There is no experimental protein expression data (MS, mRNA, etc.) for CCay to use as reference.
- **Solution**: Determine orthologous genes/proteins for all candidate proteins (with pre-existing relative protein abundance data from T. gondii in PaxDb, its closest taxonomic relative in the database) and use as a proxy to estimate SSA in CCay.

---

**Reads**:
- `data/reference/protein.faa` — CcayRef3 reference proteome
- `data/proteomes/` — full reference proteomes for T. gondii, E. tenella, P. falciparum, P. yoelii
- `data/paxdb/` — PaxDb abundance datasets

**Writes**:
- `results/orthologs.tsv` — orthogroup assignments and pairwise orthology between CCay and T. gondii.
- `results/abundance.tsv` — estimated SSA per CCay gene

---

**Resources used**:

- NCBI Datasets, T. gondii reference genome, C. cayetanensis genome.
  - O'Leary NA, Cox E, Holmes JB, Anderson WR, Falk R, Hem V, Tsuchiya MTN, Schuler GD, Zhang X, Torcivia J, Ketter A, Breen L, Cothran J, Bajwa H, Tinne J, Meric PA, Hlavina W, Schneider VA. Exploring and retrieving sequence and metadata for species across the tree of life with NCBI Datasets. Sci Data. 2024 Jul 5;11(1):732. doi: 10.1038/s41597-024-03571-y. PMID: 38969627; PMCID: PMC11226681.
- [OrthoFinder](https://orthofinder.github.io/OrthoFinder/)
  - David M Emms, Yi Liu, Laurence Belcher, Jonathan Holmes, Steven Kelly, 2025. OrthoFinder: scalable phylogenetic orthology inference for comparative genomics. bioRxiv.
- [PaxDb](https://pax-db.org/species/432359)
  - Huang, Q et al. "PaxDb v6.0: reprocessed, LLM-selected, curated protein abundance data across organisms", Nucleic Acids Research 2026, doi: 10.1093/nar/gkaf1066

## **0. Imports & installs.**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Download reference datasets from NBCI.

In [ ]:
! curl -o datasets 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets'
! chmod +x datasets

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 19.5M  100 19.5M    0     0  58.6M      0 --:--:-- --:--:-- --:--:-- 58.8M


In [ ]:
! ./datasets download genome accession GCF_002999335.1 --include gff3,rna,cds,protein,genome,seq-report --filename ccay.zip
! ./datasets download genome accession GCA_000150015.2 --include gff3,rna,cds,protein,genome,seq-report --filename tgon.zip
! ./datasets download genome accession GCF_000002765.6 --include gff3,rna,cds,protein,genome,seq-report --filename pfal.zip
! ./datasets download genome accession GCF_000003085.2 --include gff3,rna,cds,protein,genome,seq-report --filename pyoe.zip

- Collecting genome records    0 records
- Collecting genome records    0 records
- Collecting genome records    0 records
- Collecting genome records    0 records
- Collecting genome records    0 records
- Collecting genome records    0 records
\ Collecting genome records    0 records
\ Collecting genome records    0 records
\ Collecting genome records    0 records
\ Collecting genome records    0 records
\ Collecting genome records    1 record
\ Collecting genome records    1 record
\ Collecting genome records    1 record
\ Collecting genome records    1 record
\ Collecting genome records    1 record
\ Collecting genome records    1 record
\ Collecting genome records    1 record
\ Collecting genome records    1 record
\ Collecting genome records    1 record
\ Collecting genome records    1 record
| Collecting genome records    1 record
| Collecting genome records    1 record
| Collecting genome records    1 record
| Collecting genome records    1 record
| Collecting genome records   

In [ ]:
! unzip ccay.zip -d ccay
! unzip tgon.zip -d tgon
! unzip pfal.zip -d pfal
! unzip pyoe.zip -d pyoe

Archive:  ccay.zip
  inflating: ccay/README.md          
  inflating: ccay/ncbi_dataset/data/assembly_data_report.jsonl  
  inflating: ccay/ncbi_dataset/data/GCF_002999335.1/cds_from_genomic.fna  
  inflating: ccay/ncbi_dataset/data/GCF_002999335.1/GCF_002999335.1_CcayRef3_genomic.fna  
  inflating: ccay/ncbi_dataset/data/GCF_002999335.1/genomic.gff  
  inflating: ccay/ncbi_dataset/data/GCF_002999335.1/protein.faa  
  inflating: ccay/ncbi_dataset/data/GCF_002999335.1/rna.fna  
  inflating: ccay/ncbi_dataset/data/GCF_002999335.1/sequence_report.jsonl  
  inflating: ccay/ncbi_dataset/data/dataset_catalog.json  
  inflating: ccay/md5sum.txt         
Archive:  tgon.zip
  inflating: tgon/README.md          
  inflating: tgon/ncbi_dataset/data/assembly_data_report.jsonl  
  inflating: tgon/ncbi_dataset/data/GCA_000150015.2/cds_from_genomic.fna  
  inflating: tgon/ncbi_dataset/data/GCA_000150015.2/GCA_000150015.2_TGVEG_genomic.fna  
  inflating: tgon/ncbi_dataset/data/GCA_000150015.2/genomic.

Download protein abundance data from PaxDB.

In [ ]:
! curl -L https://pax-db.org/downloads/6.1/datasets/432359/432359-WHOLE_ORGANISM-integrated.txt -o t_gondii_pax.txt
! curl -L https://pax-db.org/downloads/6.1/datasets/36329/36329-WHOLE_ORGANISM-integrated.txt -o p_falciparum_pax.txt
! curl -L https://pax-db.org/downloads/6.1/datasets/73239/73239-WHOLE_ORGANISM-integrated.txt -o p_yoe_pax.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 84418    0 84418    0     0   106k      0 --:--:-- --:--:-- --:--:--  106k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  163k    0  163k    0     0   211k      0 --:--:-- --:--:-- --:--:--  211k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 45649    0 45649    0     0  69359      0 --:--:-- --:--:-- --:--:-- 69375


Install OrthoFinder and other Python imports.

In [ ]:
! pip install git+https://github.com/OrthoFinder/OrthoFinder.git

  Cloning https://github.com/OrthoFinder/OrthoFinder.git to /tmp/pip-req-build-jfw95khl
  Running command git clone --filter=blob:none --quiet https://github.com/OrthoFinder/OrthoFinder.git /tmp/pip-req-build-jfw95khl
  Resolved https://github.com/OrthoFinder/OrthoFinder.git to commit c85f0034679a9812959655f1cd60c74a30597cf1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.2/109.2 kB 11.4 MB/s eta 0:00:00
  Created wheel for orthofinder: filename=orthofinder-3.1.5.post1.dev1-py3-none-any.wh

In [ ]:
! orthofinder -h


SIMPLE USAGE:
 Run full OrthoFinder analysis on FASTA format proteomes in <dir>
   orthofinder [options] -f <dir>

 To assign species from <dir1> to existing OrthoFinder orthogroups in <dir2>
   orthofinder [options] --assign <dir1> --core <dir2>

OPTIONS:
 -t <int>                Number of parallel sequence search threads [Default =  
                         2]                                                     
 -a <int>                Number of parallel analysis threads                    
 -M <txt>                Method for gene tree inference. Options "dendroblast"  
                         & "msa" [Default = msa]                                
 -S <txt>                Sequence search program [Default = diamond]            
                         Options: diamond, diamond_ultra_sens, blastp, mmseqs,  
                         blastn                                                 
 -A <txt>                MSA program, requires "-M msa" [Default = famsa]       
             

In [ ]:
! pip install gff3-parser

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import gff3_parser as gffp
from pathlib import Path

---

## **1. Find *C. cayetanensis* orthologs.**

In [ ]:
! mkdir ortho_input

In [ ]:
! cp /content/tgon/ncbi_dataset/data/GCA_000150015.2/protein.faa /content/ortho_input/tgon.faa
! cp /content/ccay/ncbi_dataset/data/GCF_002999335.1/protein.faa /content/ortho_input/ccay.faa
! cp /content/pfal/ncbi_dataset/data/GCF_000002765.6/protein.faa /content/ortho_input/pfal.faa
! cp /content/pyoe/ncbi_dataset/data/GCF_900002385.2/protein.faa /content/ortho_input/pyoe.faa

In [ ]:
! orthofinder -f /content/ortho_input -t 2 -a 1 -M msa -A mafft -o /content/drive/MyDrive/orthofinder_out


2026-10-12 19:06:35 : Starting OrthoFinder v3.1.5.post1.dev1
2 thread(s) for highly parallel tasks (BLAST searches etc.)
1 thread(s) for OrthoFinder algorithm

OrthoFinder version 3.1.5.post1.dev1 Copyright (C) 2014 David Emms

Results directory:
    /content/orthofinder_out/Results_Oct12/

Checking required programs are installed
Test can run "mcl" - ok
Test can run "mafft" - failed

/bin/sh: 1: mafft: Permission denied

Trying OrthoFinder packaged version of MAFFT

Test can run "mafft" - ok
Test can run "fasttree" - ok

Dividing up work for BLAST for parallel processing
--------------------------------------------------
Processing... ━   4/4 0:00:00

Running diamond all-versus-all
Using 2 thread(s)
2026-10-12 19:06:36 : This may take some time...
Processing... ━   16/16 0:07:33
2026-10-12 19:14:09 : Done all-versus-all sequence search

Running OrthoFinder algorithm
-----------------------------
2026-10-12 19:14:09 : Initial processing of each species
Processing... ━   4/4 0:00:03
20

### **1.1 Map PaxDB abundance data to annotations.**

In [ ]:
orthofinder_outdir = Path('/content/drive/MyDrive/orthofinder_output')

In [ ]:
pyoe_gff = gffp.parse_gff3('/content/pyoe/ncbi_dataset/data/GCF_000003085.2/genomic.gff', parse_attributes=True)
pyoe_gff = pyoe_gff[pyoe_gff['Type'] == 'CDS'] # subset to coding sequences only

pyoe_cols = ['gene_name', 'string_external_id', 'abundance']
pyoe_pax = pd.read_csv('/content/p_yoe_pax.txt', sep='\t', comment='#', names=pyoe_cols)

!genome-build-accession NCBI_Assembly:GCF_000003085.2
species https://www.ncbi.nlm.nih.gov/Taxonomy/Browser/wwwtax.cgi?id=352914
Building structured data...


100%|██████████| 150162/150162 [00:00<00:00, 614338.22it/s]
/usr/local/lib/python3.13/dist-packages/gff3_parser/parser.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = pd.DataFrame(data).replace('.', np.nan)


Adding Supplemental Attribute table...
Finding unique attribute keys...


100%|██████████| 138782/138782 [00:00<00:00, 528319.91it/s]


Making attribute table...


100%|██████████| 138782/138782 [00:01<00:00, 114234.82it/s]


In [ ]:
print(pyoe_pax.shape)
print(pyoe_pax.gene_name.nunique())

(1768, 3)
1768


In [ ]:
def map_abundances(gff_path, pax_path, gff_subset='CDS'):
  gff = gffp.parse_gff3(gff_path, parse_attributes=True)
  print(gff.columns)
  gff = gff[gff['Type'] == gff_subset] # subset to coding sequences only

  cols = ['gene_name', 'string_external_id', 'abundance']
  pax = pd.read_csv(pax_path, sep='\t', comment='#', names=cols)

  no_pax = [no_pax for no_pax in gff.locus_tag.unique() if no_pax not in pax.gene_name.unique()]
  print(len(no_pax))

  abundances = pd.merge(gff, pax, left_on='locus_tag', right_on='gene_name', how='left')
  abundances = abundances[abundances.abundance.notna()]
  print(abundances.shape)

  pax_map = {}
  if gff_subset == 'CDS':
    pax_map = {row.protein_id: row.abundance for row in abundances.itertuples()}
  else:
    pax_map = {row.locus_tag: row.abundance for row in abundances.itertuples()}

  return pax_map, abundances


### **Why does *P. yoelii* use 'gene' instead of 'CDS'?**

The reference genome for *P. yoelii* **does** have CDS features annotated. However, in its GFF file, the `locus_tag` attribute is completely missing from the `CDS` rows, and is only present on the `gene` rows.

Because `map_abundances` merges the GFF annotations with the PaxDB data using the `locus_tag` field, we must pass `gff_subset='gene'` for *P. yoelii* to successfully match the PaxDB IDs (e.g., `PY00001`). The subsequent cells then map these gene IDs back to their corresponding `protein_id`s so they match the OrthoFinder output.

In [ ]:
pfal_hash, pfal_abundances = map_abundances('/content/pfal/ncbi_dataset/data/GCF_000002765.6/genomic.gff', '/content/p_falciparum_pax.txt')
pyoe_hash_tmp, pyoe_abundances = map_abundances('/content/pyoe/ncbi_dataset/data/GCF_000003085.2/genomic.gff', '/content/p_yoe_pax.txt', 'gene')
tgon_hash, tgon_abundances = map_abundances('/content/tgon/ncbi_dataset/data/GCA_000150015.2/genomic.gff', '/content/t_gondii_pax.txt')

!genome-build-accession NCBI_Assembly:GCF_000002765.6
species https://www.ncbi.nlm.nih.gov/Taxonomy/Browser/wwwtax.cgi?id=36329
Building structured data...


100%|██████████| 40414/40414 [00:00<00:00, 284346.48it/s]
/usr/local/lib/python3.13/dist-packages/gff3_parser/parser.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = pd.DataFrame(data).replace('.', np.nan)


Adding Supplemental Attribute table...
Finding unique attribute keys...


100%|██████████| 40378/40378 [00:00<00:00, 258650.87it/s]


Making attribute table...


100%|██████████| 40378/40378 [00:00<00:00, 80523.85it/s]


... (9 outputs left out)

100%|██████████| 113466/113466 [00:00<00:00, 224949.75it/s]


Making attribute table...


100%|██████████| 113466/113466 [00:02<00:00, 45742.48it/s]


Index(['Seqid', 'Source', 'Type', 'Start', 'End', 'Score', 'Strand', 'Phase',
       'Parent', 'genome', 'protein_id', 'inference', 'go_function',
       'end_range', 'orig_transcript_id', 'start_range', 'ID', 'Dbxref',
       'gbkey', 'partial', 'Name', 'Ontology_term', 'product', 'locus_tag',
       'strain', 'map', 'orig_protein_id', 'gene_biotype', 'chromosome',
       'go_process', 'Note', 'mol_type', 'go_component'],
      dtype='object')
7747
(5081, 36)


In [ ]:
# post-hoc remap pyoe gene names to protein_id
pyoe_gff = gffp.parse_gff3('/content/pyoe/ncbi_dataset/data/GCF_000003085.2/genomic.gff', parse_attributes=True)
pyoe_gff = pyoe_gff[(pyoe_gff['Type'] == 'CDS') | (pyoe_gff['Type'] == 'gene')]
pyoe_gff.head(15)

!genome-build-accession NCBI_Assembly:GCF_000003085.2
species https://www.ncbi.nlm.nih.gov/Taxonomy/Browser/wwwtax.cgi?id=352914
Building structured data...


100%|██████████| 150162/150162 [00:00<00:00, 626311.47it/s]
/usr/local/lib/python3.13/dist-packages/gff3_parser/parser.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = pd.DataFrame(data).replace('.', np.nan)


Adding Supplemental Attribute table...
Finding unique attribute keys...


100%|██████████| 138782/138782 [00:00<00:00, 521987.08it/s]


Making attribute table...


100%|██████████| 138782/138782 [00:01<00:00, 120190.01it/s]


,Seqid,Source,Type,Start,End,Score,Strand,Phase,old-name,Parent,genome,gene_synonym,protein_id,transcript_id,rpt_family,end_range,sub-species,start_range,ID,Dbxref,exon_number,gbkey,gene,partial,Name,isolate,product,locus_tag,strain,gene_biotype,Note,mol_type
7,NW_862602.3,RefSeq,gene,1173,2212,NaN,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gene0,GeneID:3748341,NaN,Gene,NaN,NaN,PY00001,NaN,NaN,PY00001,NaN,protein_coding,NaN,NaN
11,NW_862602.3,RefSeq,CDS,1173,1187,NaN,+,0,NaN,rna0,NaN,NaN,XP_723613.1,NaN,NaN,NaN,NaN,NaN,cds0,"Genbank:XP_723613.1,GeneID:3748341",NaN,CDS,NaN,NaN,XP_723613.1,NaN,hypothetical protein,NaN,NaN,NaN,NaN,NaN
12,NW_862602.3,RefSeq,CDS,1322,2212,NaN,+,0,NaN,rna0,NaN,NaN,XP_723613.1,NaN,NaN,NaN,NaN,NaN,cds0,"Genbank:XP_723613.1,GeneID:3748341",NaN,CDS,NaN,NaN,XP_723613.1,NaN,hypothetical protein,NaN,NaN,NaN,NaN,NaN
16,NW_862602.3,RefSeq,gene,2357,2430,NaN,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2430,.",NaN,NaN,gene1,GeneID:3748342,NaN,Gene,NaN,true,PY00002,NaN,NaN,PY00002,NaN,protein_coding,NaN,NaN
19,NW_862602.3,RefSeq,CDS,2357,2430,NaN,+,0,NaN,rna1,NaN,NaN,XP_723614.1,NaN,NaN,"2430,.",NaN,NaN,cds1,"Genbank:XP_723614.1,GeneID:3748342",NaN,CDS,NaN,true,XP_723614.1,NaN,hypothetical protein,NaN,NaN,NaN,NaN,NaN
27,NW_862603.3,RefSeq,gene,875,1513,NaN,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gene2,GeneID:3431723,NaN,Gene,NaN,NaN,PY00048,NaN,NaN,PY00048,NaN,protein_coding,NaN,NaN
30,NW_862603.3,RefSeq,CDS,875,1513,NaN,+,0,NaN,rna2,NaN,NaN,XP_723615.1,NaN,NaN,NaN,NaN,NaN,cds2,"Genbank:XP_723615.1,GeneID:3431723",NaN,CDS,NaN,NaN,XP_723615.1,NaN,hypothetical protein,NaN,NaN,NaN,NaN,NaN
46,NW_862604.3,RefSeq,gene,1563,5147,NaN,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gene3,GeneID:3431724,NaN,Gene,NaN,NaN,PY00348,NaN,NaN,PY00348,NaN,protein_coding,NaN,NaN
49,NW_862604.3,RefSeq,CDS,1563,5147,NaN,-,0,NaN,rna3,NaN,NaN,XP_723616.1,NaN,NaN,NaN,NaN,NaN,cds3,"Genbank:XP_723616.1,GeneID:3431724",NaN,CDS,NaN,NaN,XP_723616.1,NaN,CCAAT-box DNA binding protein subunit B,NaN,NaN,NaN,NaN,NaN
92,NW_862605.3,RefSeq,gene,926,2587,NaN,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gene4,GeneID:3431725,NaN,Gene,NaN,NaN,PY03465,NaN,NaN,PY03465,NaN,protein_coding,NaN,NaN


In [ ]:
print(pyoe_gff[pyoe_gff['Type'] == 'gene'].Name.nunique())
print(pyoe_gff[pyoe_gff['Type'] == 'CDS'].Name.nunique())

7403
7353


In [ ]:
# create a GeneID column from Dbxref
for row in pyoe_gff.itertuples():
  references = row.Dbxref.split(',')
  for ref in references:
    if ref.startswith('GeneID:'):
      pyoe_gff.at[row.Index, 'GeneID'] = ref.split(':')[1]

In [ ]:
pyoe_gff.head()

,Seqid,Source,Type,Start,End,Score,Strand,Phase,old-name,Parent,genome,gene_synonym,protein_id,transcript_id,rpt_family,end_range,sub-species,start_range,ID,Dbxref,exon_number,gbkey,gene,partial,Name,isolate,product,locus_tag,strain,gene_biotype,Note,mol_type,GeneID
7,NW_862602.3,RefSeq,gene,1173,2212,NaN,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gene0,GeneID:3748341,NaN,Gene,NaN,NaN,PY00001,NaN,NaN,PY00001,NaN,protein_coding,NaN,NaN,3748341
11,NW_862602.3,RefSeq,CDS,1173,1187,NaN,+,0,NaN,rna0,NaN,NaN,XP_723613.1,NaN,NaN,NaN,NaN,NaN,cds0,"Genbank:XP_723613.1,GeneID:3748341",NaN,CDS,NaN,NaN,XP_723613.1,NaN,hypothetical protein,NaN,NaN,NaN,NaN,NaN,3748341
12,NW_862602.3,RefSeq,CDS,1322,2212,NaN,+,0,NaN,rna0,NaN,NaN,XP_723613.1,NaN,NaN,NaN,NaN,NaN,cds0,"Genbank:XP_723613.1,GeneID:3748341",NaN,CDS,NaN,NaN,XP_723613.1,NaN,hypothetical protein,NaN,NaN,NaN,NaN,NaN,3748341
16,NW_862602.3,RefSeq,gene,2357,2430,NaN,+,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2430,.",NaN,NaN,gene1,GeneID:3748342,NaN,Gene,NaN,true,PY00002,NaN,NaN,PY00002,NaN,protein_coding,NaN,NaN,3748342
19,NW_862602.3,RefSeq,CDS,2357,2430,NaN,+,0,NaN,rna1,NaN,NaN,XP_723614.1,NaN,NaN,"2430,.",NaN,NaN,cds1,"Genbank:XP_723614.1,GeneID:3748342",NaN,CDS,NaN,true,XP_723614.1,NaN,hypothetical protein,NaN,NaN,NaN,NaN,NaN,3748342


In [ ]:
# group by End
pyoe_gff_grouped = {name: group for name, group in pyoe_gff.groupby('GeneID')}

In [ ]:
pyoe_g2p = {}
for group in pyoe_gff_grouped.values():
  if len(group) > 1:
    # create a set from the group.Name
    names = sorted(set(group.Name))
    pyoe_g2p[names[0]] = names[1]
pyoe_g2p
len(pyoe_g2p.keys())

7353

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
# create properly remapped pyoe_hash
pyoe_hash = {}
for k, v in pyoe_hash_tmp.items():
  if k in pyoe_g2p:
    pyoe_hash[pyoe_g2p[k]] = v
pyoe_hash


{'XP_723619.1': 23.1,
 'XP_723620.1': 7.56,
 'XP_723621.1': 14.0,
 'XP_723622.1': 18.0,
 'XP_723625.1': 104.0,
 'XP_723628.1': 205.0,
 'XP_723629.1': 367.0,
 'XP_723630.1': 398.0,
 'XP_723631.1': 382.0,
 'XP_723636.1': 17.5,
 'XP_723641.1': 9.02,
 'XP_723644.1': 28.2,
 'XP_723647.1': 5.17,
 'XP_723652.1': 81.7,
 'XP_723655.1': 477.0,
 'XP_723656.1': 24.2,
 'XP_723657.1': 521.0,
 'XP_723658.1': 327.0,
 'XP_723664.1': 691.0,
 'XP_723666.1': 495.0,
 'XP_723667.1': 97.4,
 'XP_723676.1': 17.3,
 'XP_723677.1': 21.1,
 'XP_723678.1': 39.1,
 'XP_723680.1': 642.0,
 'XP_723684.1': 1403.0,
 'XP_723685.1': 2.25,
 'XP_723687.1': 116.0,
 'XP_723688.1': 2445.0,
 'XP_723691.1': 3.66,
 'XP_723704.1': 2.34,
 'XP_723708.1': 802.0,
 'XP_723710.1': 2.73,
 'XP_723713.1': 361.0,
 'XP_723714.1': 360.0,
 'XP_723718.1': 10.2,
 'XP_723722.1': 392.0,
 'XP_723737.1': 8622.0,
 'XP_723743.1': 121.0,
 'XP_723746.1': 130.0,
 'XP_723747.1': 97.2,
 'XP_723750.1': 35.3,
 'XP_723759.1': 10.2,
 'XP_723760.1': 1747.0,
 'XP_7

### **1.2 Map OrthoGroups to annotations from taxonomic relatives.**

First, let's ignore any orthogroups without any CCay genes.

In [ ]:
orthogroups_genecounts = pd.read_csv('/content/drive/MyDrive/orthofinder_out/Results_Oct12/Orthogroups/Orthogroups.GeneCount.tsv', sep='\t')
orthogroups_genecounts = orthogroups_genecounts[orthogroups_genecounts.ccay != 0]
orthogroups_genecounts.shape

(3982, 6)

In [ ]:
orthogroups = pd.read_csv('/content/drive/MyDrive/orthofinder_out/Results_Oct12/Orthogroups/Orthogroups.tsv', sep='\t')
orthogroups = orthogroups[orthogroups.ccay.notna()]
print(orthogroups.shape)

orthogroups.head()

(3982, 5)


,Orthogroup,ccay,pfal,pyoe,tgon
12,OG0000012,"XP_022592735.1, XP_026190321.1, XP_026192749.1",NaN,NaN,"ESS28122.1, ESS28125.1, ESS28165.1, ESS28175.1..."
15,OG0000015,XP_026191021.1,"XP_001347662.1, XP_001347707.1, XP_001347710.1...",XP_730811.1,ESS35076.1
16,OG0000016,"XP_022590852.2, XP_022592567.2, XP_022593424.2...","XP_001348028.1, XP_001348410.1, XP_001350297.1...","XP_022811634.1, XP_723634.1, XP_727062.1, XP_7...","ESS28613.1, ESS28795.1, ESS28827.1, ESS28832.1..."
22,OG0000022,XP_022586852.1,"XP_001347300.2, XP_001348170.1, XP_001348925.1...",XP_729448.2,ESS34524.1
23,OG0000023,"XP_022589344.1, XP_022589794.1, XP_026191254.1...","XP_001347663.1, XP_001348190.1, XP_001348911.1...","XP_022811496.1, XP_034493488.1, XP_725876.2, X...","ESS28591.1, ESS28974.1, ESS29047.1, ESS30683.1..."


In [ ]:
ccay_protein_orthologs = pd.read_csv('/content/drive/MyDrive/orthofinder_out/Results_Oct12/Orthologues/ccay.tsv', sep='\t')
ccay_protein_orthologs.head()

,Orthogroup,Species,ccay,Orthologs
0,OG0000012,tgon,XP_026190321.1,ESS31854.1
1,OG0000012,tgon,"XP_022592735.1, XP_026192749.1","ESS35073.1, ESS28175.1, ESS31519.1, ESS28122.1..."
2,OG0000015,pyoe,XP_026191021.1,XP_730811.1
3,OG0000015,pfal,XP_026191021.1,"XP_001349457.1, XP_002808892.1, XP_001347710.1..."
4,OG0000015,tgon,XP_026191021.1,ESS35076.1


In [ ]:
species_to_hash = {'tgon': tgon_hash, 'pfal': pfal_hash, 'pyoe': pyoe_hash}

ccay_protein_orthologs_mapped = ccay_protein_orthologs.copy()
# Initialize column with object dtype to hold list values
ccay_protein_orthologs_mapped['abundance'] = [[] for _ in range(len(ccay_protein_orthologs_mapped))]

for row in ccay_protein_orthologs_mapped.itertuples():
  abundances = []
  # Strip any potential leading/trailing whitespace from the split ortholog IDs
  orthologs = [o.strip() for o in row.Orthologs.split(',')]
  species = row.Species
  for ortholog in orthologs:
    if ortholog in species_to_hash[species]:
      abundances.append(species_to_hash[species][ortholog])
    else:
      abundances.append(np.nan)

  # set a list value into a single cell
  ccay_protein_orthologs_mapped.at[row.Index, 'abundance'] = abundances

ccay_protein_orthologs_mapped.head()

,Orthogroup,Species,ccay,Orthologs,abundance
0,OG0000012,tgon,XP_026190321.1,ESS31854.1,[8.78]
1,OG0000012,tgon,"XP_022592735.1, XP_026192749.1","ESS35073.1, ESS28175.1, ESS31519.1, ESS28122.1...","[nan, 175.0, 9.65, 1.51, 2.49, 2.04, 0.514, na..."
2,OG0000015,pyoe,XP_026191021.1,XP_730811.1,[37.1]
3,OG0000015,pfal,XP_026191021.1,"XP_001349457.1, XP_002808892.1, XP_001347710.1...","[183.0, 0.228, 0.018, 0.511, nan, 0.514, 1.6, ..."
4,OG0000015,tgon,XP_026191021.1,ESS35076.1,[nan]


In [ ]:
# print all rows with at least 1 non-na abundance
ccay_protein_orthologs_mapped[ccay_protein_orthologs_mapped.abundance.apply(lambda x: not all(pd.isna(a) for a in x))]

""


## **2. Saving results & cursory analysis.**

In [ ]:
ccay_protein_orthologs_mapped.to_csv('/content/drive/MyDrive/ccay_abundance.tsv', sep='\t', index=False)

Some napkin analysis:

In [ ]:
# let's subset to species==tgon and len of orthologs ccay and Orthologs is 1
ccay_tgon = ccay_protein_orthologs_mapped[ccay_protein_orthologs_mapped.Species == 'tgon']
ccay_tgon = ccay_tgon[ccay_tgon.Orthologs.apply(lambda x: len(x.split(',')) == 1)]
ccay_tgon = ccay_tgon[ccay_tgon.ccay.apply(lambda x: len(x.split(',')) == 1)]

# sort by abundance, first unpacking list
ccay_tgon['abundance'] = ccay_tgon['abundance'].apply(lambda x: np.mean(x))
ccay_tgon = ccay_tgon.sort_values(by='abundance', ascending=False)

# drop rows with nan abundance
ccay_tgon = ccay_tgon.dropna(subset=['abundance'])

ccay_tgon.head(10)

,Orthogroup,Species,ccay,Orthologs,abundance
3006,OG0001002,tgon,XP_022590868.1,ESS29264.1,7120.0
2630,OG0000873,tgon,XP_022587857.1,ESS31933.1,2825.0
4936,OG0001652,tgon,XP_026191360.1,ESS35745.1,2661.0
4079,OG0001365,tgon,XP_026190457.1,ESS34294.1,2616.0
3246,OG0001082,tgon,XP_022592765.1,ESS35066.1,2437.0
7180,OG0002405,tgon,XP_026193719.1,ESS29972.1,2421.0
6869,OG0002301,tgon,XP_026193408.1,ESS33818.1,2228.0
1870,OG0000611,tgon,XP_026193152.1,ESS35974.1,2131.0
6051,OG0002027,tgon,XP_026192565.1,ESS35402.1,2084.0
210,OG0000076,tgon,XP_022587647.1,ESS36403.1,2056.0


In [ ]:
ccay_pfal = ccay_protein_orthologs_mapped[ccay_protein_orthologs_mapped.Species == 'pfal']
ccay_pfal = ccay_pfal[ccay_pfal.Orthologs.apply(lambda x: len(x.split(',')) == 1)]
ccay_pfal = ccay_pfal[ccay_pfal.ccay.apply(lambda x: len(x.split(',')) == 1)]

# sort by abundance, first unpacking list
ccay_pfal['abundance'] = ccay_pfal['abundance'].apply(lambda x: np.mean(x))
ccay_pfal = ccay_pfal.sort_values(by='abundance', ascending=False)

# drop rows with nan abundance
ccay_pfal = ccay_pfal.dropna(subset=['abundance'])

ccay_pfal.head(10)

,Orthogroup,Species,ccay,Orthologs,abundance
3211,OG0001071,pfal,XP_022592569.1,XP_088452675.1,15999.0
1178,OG0000359,pfal,XP_026190421.1,XP_001349989.1,7918.0
1172,OG0000357,pfal,XP_022589061.1,XP_088452676.1,7871.0
2843,OG0000947,pfal,XP_022589497.1,XP_001351592.1,6315.0
2210,OG0000729,pfal,XP_022590042.1,XP_001348542.1,6071.0
1970,OG0000648,pfal,XP_022589115.1,XP_001348998.1,6068.0
2102,OG0000693,pfal,XP_026190337.1,XP_001352050.1,5854.0
1652,OG0000523,pfal,XP_026193302.1,XP_001347440.1,5736.0
2060,OG0000679,pfal,XP_026193429.1,XP_088452833.1,5655.0
1278,OG0000393,pfal,XP_022592409.1,XP_088452835.1,4959.0


In [ ]:
ccay_pyoe = ccay_protein_orthologs_mapped[ccay_protein_orthologs_mapped.Species == 'pyoe']
ccay_pyoe = ccay_pyoe[ccay_pyoe.Orthologs.apply(lambda x: len(x.split(',')) == 1)]
ccay_pyoe = ccay_pyoe[ccay_pyoe.ccay.apply(lambda x: len(x.split(',')) == 1)]

# sort by abundance, first unpacking list
ccay_pyoe['abundance'] = ccay_pyoe['abundance'].apply(lambda x: np.mean(x))
ccay_pyoe = ccay_pyoe.sort_values(by='abundance', ascending=False)

# drop rows with nan abundance
ccay_pyoe = ccay_pyoe.dropna(subset=['abundance'])

ccay_pyoe.head(10)

,Orthogroup,Species,ccay,Orthologs,abundance
2061,OG0000679,pyoe,XP_026193429.1,XP_724814.1,13244.0
4315,OG0001445,pyoe,XP_026190741.1,XP_728795.1,5073.0
1175,OG0000358,pyoe,XP_022588287.1,XP_726754.1,4405.0
1312,OG0000404,pyoe,XP_026189744.1,XP_730727.1,3624.0
1279,OG0000393,pyoe,XP_022592409.1,XP_725477.1,3518.0
1179,OG0000359,pyoe,XP_026190421.1,XP_724101.1,3460.0
2599,OG0000863,pyoe,XP_022587744.1,XP_731151.1,3439.0
4692,OG0001571,pyoe,XP_026191109.1,XP_726653.1,3203.0
4206,OG0001408,pyoe,XP_026190612.1,XP_727122.1,3118.0
2091,OG0000689,pyoe,XP_026190942.1,XP_725803.1,3079.0


### **2.1 Subsetting to extracellular proteins (DeepLoc2.1).**

In [ ]:
locs = pd.read_csv('/content/results_20261008-181607.csv') # from notebook CCayBinder_02
locs.head()

,Protein_ID,Localizations,Signals,Membrane types,Cytoplasm,Nucleus,Extracellular,Cell membrane,Mitochondrion,Plastid,Endoplasmic reticulum,Lysosome/Vacuole,Golgi apparatus,Peroxisome,Peripheral,Transmembrane,Lipid anchor,Soluble
0,XP_022586349.2,Endoplasmic reticulum,Signal peptide|Transmembrane domain,Transmembrane,0.1282,0.1820,0.0449,0.3204,0.1403,0.0166,0.8768,0.3592,0.4063,0.0034,0.0602,0.9968,0.0733,0.0925
1,XP_022586352.2,Cytoplasm,Nuclear localization signal,Soluble,0.6825,0.5030,0.0695,0.2695,0.1132,0.0119,0.0488,0.1567,0.2283,0.0216,0.5857,0.0544,0.0791,0.7674
2,XP_022586359.2,Cytoplasm,Nuclear export signal,Soluble,0.5634,0.4525,0.0260,0.2451,0.3105,0.0432,0.1402,0.3217,0.3660,0.0141,0.6064,0.0998,0.0761,0.6501
3,XP_022586363.2,Endoplasmic reticulum,Signal peptide,Soluble,0.1384,0.0863,0.5286,0.1683,0.1092,0.0751,0.5906,0.5054,0.4678,0.0087,0.4487,0.1352,0.0294,0.6851
4,XP_022586398.1,Cytoplasm|Nucleus,NaN,Soluble,0.5415,0.5831,0.2129,0.0653,0.2987,0.0305,0.1886,0.0914,0.2443,0.0206,0.5487,0.0857,0.1642,0.7805


In [ ]:
# check how many rows where the extracellular prediction > 80%
(len(locs.Extracellular[locs.Extracellular >= 0.80]))

137

In [ ]:
locs.value_counts('Localizations')

,count
Localizations,
Cytoplasm,1807
Nucleus,1024
Cytoplasm|Nucleus,646
Golgi apparatus,596
Mitochondrion,453
Extracellular,318
Endoplasmic reticulum,217
Cell membrane,127
Cytoplasm|Golgi apparatus,99


In [ ]:
ec_proteins = locs[locs.Extracellular >= 0.80]

In [ ]:
# let's see which proteins from loc have abundance data

print("T. gondii", len([x for x in ec_proteins.Protein_ID if x in ccay_tgon.ccay.values]) / len(ec_proteins.Protein_ID) * 100, "%")
print("P. falciparum", len([x for x in ec_proteins.Protein_ID if x in ccay_pfal.ccay.values]) / len(ec_proteins.Protein_ID) * 100, "%")
print("P. yoelli: ",len([x for x in ec_proteins.Protein_ID if x in ccay_pyoe.ccay.values]) / len(ec_proteins.Protein_ID) * 100, "%")

T. gondii 2.18978102189781 %
P. falciparum 8.02919708029197 %
P. yoelli:  1.4598540145985401 %
